# Fink Tag Dataset Browser

Browse and visualise a dataset downloaded by `fink_download_tag_dataset.py`.

Expected layout on disk:
```
fink_dataset/
└── <TAG_CONFIG>/
    ├── catalog.parquet
    ├── light_curves/
    │   └── lc_<diaObjectId>.parquet
    └── cutouts/
        └── cutout_<diaObjectId>.npy
```

**Column naming reminder (LSST DPDD schema)**
- `r:<col>` — diaSource table field. The `r:` prefix is the **table name**, NOT the spectral band.
- `f:<col>` — Fink-computed field (classifiers, cross-matches).
- Spectral band → value of `r:band` ∈ {u, g, r, i, z, y}.

## 1 — Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from astropy.visualization import ZScaleInterval

# Reuse the plot helpers from fink_alert_lib (same directory)
from rubinlsstskyalerts.fink_tools import (
    BAND_COLORS,
    BAND_WAVELENGTHS,
    RUBIN_ZEROPOINT,
    DARK_BG, PANEL_BG, TEXT_COL, MUTED_COL, ACCENT, HIGHLIGHT, BORDER,
    flux_to_mag,
    plot_lightcurve_flux,
    plot_lightcurve_mag,
    plot_cutouts,
    plot_classifiers,
)

#%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Imports OK")

## 2 — Dataset loader for Pipeline B layout

This loader handles the directory structure produced by `fink_download_tag_dataset.py`,
which differs from the Pipeline A layout expected by `FinkDataset` in `fink_alert_lib.py`.

In [ ]:
class TagDataset:
    """
    Loader for a single-tag dataset produced by fink_download_tag_dataset.py.

    Directory layout (under tag_dir):
        catalog.parquet
        light_curves/lc_<diaObjectId>.parquet
        cutouts/cutout_<diaObjectId>.npy   (dict: Science/Template/Difference)

    Parameters
    ----------
    tag_dir : Path
        Path to the tag sub-directory, e.g. fink_dataset/extragalactic_new_candidate/
    """

    def __init__(self, tag_dir: Path) -> None:
        self.tag_dir = Path(tag_dir)
        self.lc_dir = self.tag_dir / "light_curves"
        self.cutout_dir = self.tag_dir / "cutouts"

        # Load catalog (one row per diaObject, most recent diaSource)
        cat_path = self.tag_dir / "catalog.parquet"
        assert cat_path.exists(), f"catalog.parquet not found in {self.tag_dir}"
        self.catalog = pd.read_parquet(cat_path)

        # Build sorted list of diaObjectIds
        self.obj_ids = sorted(self.catalog["r:diaObjectId"].unique().tolist())

    # ── Accessors ─────────────────────────────────────────────────────────────

    def __len__(self) -> int:
        return len(self.obj_ids)

    def get_meta(self, obj_id: int) -> pd.Series:
        """Return the catalog row for a given diaObjectId."""
        rows = self.catalog[self.catalog["r:diaObjectId"] == obj_id]
        if rows.empty:
            raise KeyError(f"diaObjectId {obj_id} not found in catalog.")
        return rows.iloc[0]

    def get_lightcurve(self, obj_id: int) -> pd.DataFrame:
        """Load the light curve parquet for a given diaObjectId."""
        path = self.lc_dir / f"lc_{obj_id}.parquet"
        if not path.exists():
            return pd.DataFrame()
        df = pd.read_parquet(path)
        if "r:midpointMjdTai" in df.columns:
            df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)
        return df

    def get_cutouts(self, obj_id: int) -> dict | None:
        """
        Load cutout dict for a given diaObjectId.

        Returns
        -------
        dict with keys 'Science', 'Template', 'Difference', each a 2-D float32 array,
        or None if not available.
        """
        path = self.cutout_dir / f"cutout_{obj_id}.npy"
        if not path.exists():
            return None
        data = np.load(path, allow_pickle=True).item()  # dict saved with np.save
        return data

    def get_cutout_array(self, obj_id: int) -> np.ndarray | None:
        """
        Return cutouts as a stacked (3, H, W) array [Science, Template, Difference].
        Compatible with plot_cutouts() from fink_alert_lib.
        """
        d = self.get_cutouts(obj_id)
        if d is None:
            return None
        return np.stack([d["Science"], d["Template"], d["Difference"]], axis=0)

    def summary(self) -> None:
        """Print a summary of the dataset."""
        print(f"Tag directory : {self.tag_dir.resolve()}")
        print(f"Objects       : {len(self.obj_ids)}")
        print(f"Light curves  : {len(list(self.lc_dir.glob('*.parquet')))}")
        print(f"Cutouts       : {len(list(self.cutout_dir.glob('*.npy')))}")
        print(f"\nCatalog columns:")
        for c in self.catalog.columns:
            print(f"  {c}")

print("TagDataset class defined.")

## 3 — Choose the tag and load the dataset

Set `TAG_CONFIG` to the tag you downloaded with `fink_download_tag_dataset.py`.

       TAGS_CONFIG = {
        "extragalactic_new_candidate": 1,  # nouveau (< 48h) + extragalactique
        "extragalactic_lt20mag_candidate": 1,  # rising, bright (mag < 20) +   extragalactique
        "sn_near_galaxy_candidate": 1,  # proche galaxie + SNe-like
        "in_tns": 1,  # contrepartie connue dans TNS
        "hostless_candidate": 0,  # sans hôte (ELEPHANT)
        }

In [ ]:
# ── USER PARAMETERS ─────────────────────────────────────────────────────────

#TAG_CONFIG = "extragalactic_new_candidate"   # short tag name used in --tag
#TAG_CONFIG = "extragalactic_lt20mag_candidate"
TAG_CONFIG = "sn_near_galaxy_candidate"
BASE_DIR   = Path("fink_dataset")            # base output directory of the script

# ─────────────────────────────────────────────────────────────────────────────

tag_dir = BASE_DIR / TAG_CONFIG
ds = TagDataset(tag_dir)
ds.summary()

## 4 — Explore the catalog

In [ ]:
# Full catalog table
print(f"{len(ds.catalog)} rows × {len(ds.catalog.columns)} columns")
ds.catalog.head(10)

In [ ]:
# List of available diaObjectIds
print(f"{len(ds.obj_ids)} objects:")
print(ds.obj_ids[:20], "..." if len(ds.obj_ids) > 20 else "")

In [ ]:
# Classifier score distribution
score_col = "f:clf_snnSnVsOthers_score"
if score_col in ds.catalog.columns:
    scores = ds.catalog[score_col].dropna()
    fig, ax = plt.subplots(figsize=(7, 3), facecolor=DARK_BG)
    ax.set_facecolor(PANEL_BG)
    ax.hist(scores, bins=30, color=ACCENT, edgecolor=DARK_BG, linewidth=0.5)
    ax.axvline(0.5, color=HIGHLIGHT, lw=1.5, ls="--", label="threshold 0.5")
    ax.set_xlabel("SNN score (SN vs others)", color=MUTED_COL)
    ax.set_ylabel("Count", color=MUTED_COL)
    ax.set_title(f"{TAG_CONFIG}  —  score distribution  (N={len(scores)})", color=ACCENT)
    ax.tick_params(colors=MUTED_COL)
    ax.legend(fontsize=8, facecolor=DARK_BG, labelcolor=TEXT_COL)
    plt.tight_layout()
    plt.show()

In [ ]:
# Sky distribution (RA / Dec scatter)
fig, ax = plt.subplots(figsize=(8, 4), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)
sc = ax.scatter(
    ds.catalog["r:ra"],
    ds.catalog["r:dec"],
    c=ds.catalog.get(score_col, pd.Series(dtype=float)),
    cmap="plasma",
    s=20,
    alpha=0.8,
)
plt.colorbar(sc, ax=ax, label="SNN score")
ax.set_xlabel("RA (deg)", color=MUTED_COL)
ax.set_ylabel("Dec (deg)", color=MUTED_COL)
ax.set_title(f"{TAG_CONFIG}  —  sky distribution", color=ACCENT)
ax.tick_params(colors=MUTED_COL)
plt.tight_layout()
plt.show()

## 5 — Select an individual object

Set `OBJ_INDEX` (0-based position in the list) **or** paste an `OBJ_ID` directly.

In [ ]:
# ── SELECTION ────────────────────────────────────────────────────────────────

OBJ_INDEX = 0       # 0-based index into ds.obj_ids
OBJ_ID    = None    # set to an explicit diaObjectId to override OBJ_INDEX

# ─────────────────────────────────────────────────────────────────────────────

if OBJ_ID is None:
    OBJ_ID = ds.obj_ids[OBJ_INDEX]

meta = ds.get_meta(OBJ_ID)
print(f"Selected diaObjectId : {OBJ_ID}")
print(f"  RA={meta['r:ra']:.5f}  Dec={meta['r:dec']:.5f}")
print(f"  Band={meta.get('r:band','?')}  SNR={meta.get('r:snr', float('nan')):.1f}")
print(f"  SNN score = {meta.get('f:clf_snnSnVsOthers_score', float('nan')):.4f}")
print(f"  TNS name  = {meta.get('f:xm_tns_fullname', '—')}")
print(f"  TNS type  = {meta.get('f:xm_tns_type', '—')}")
print(f"  SIMBAD    = {meta.get('f:xm_simbad_otype', '—')}")
print(f"  zphot     = {meta.get('f:xm_legacydr8_zphot', '—')}")

## 6 — Light curves

In [ ]:
lc = ds.get_lightcurve(OBJ_ID)
print(f"{len(lc)} diaSources, bands: {sorted(lc['r:band'].unique())}")
lc.head()

In [ ]:
# Flux light curve
fig, ax = plt.subplots(figsize=(9, 4), facecolor=DARK_BG)
plot_lightcurve_flux(lc, ax=ax, title=f"Flux LC — diaObjectId {OBJ_ID}")
plt.tight_layout()
plt.show()

In [ ]:
# Magnitude light curve
fig, ax = plt.subplots(figsize=(9, 4), facecolor=DARK_BG)
plot_lightcurve_mag(lc, ax=ax, title=f"Mag AB LC — diaObjectId {OBJ_ID}")
plt.tight_layout()
plt.show()

## 7 — Cutout stamps

In [ ]:
cutouts_dict = ds.get_cutouts(OBJ_ID)
cutouts_arr  = ds.get_cutout_array(OBJ_ID)   # shape (3, H, W)

if cutouts_arr is not None:
    print(f"Cutout shape : {cutouts_arr.shape}")
    for k, v in cutouts_dict.items():
        print(f"  {k:12s}  min={v.min():.1f}  max={v.max():.1f}  std={v.std():.1f}")
else:
    print("No cutout available for this object.")

In [ ]:
# Plot the three stamps
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor=DARK_BG)
plot_cutouts(cutouts_arr, band=meta.get("r:band", "?"), axes=list(axes))
fig.suptitle(f"Cutouts — diaObjectId {OBJ_ID}", color=TEXT_COL, fontsize=11,
             fontfamily="monospace", y=1.01)
plt.tight_layout()
plt.show()

## 8 — Classifier scores

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), facecolor=DARK_BG)
plot_classifiers(meta, ax=ax)
ax.set_title(f"Classifiers — diaObjectId {OBJ_ID}", color=ACCENT)
plt.tight_layout()
plt.show()

## 9 — Full overview (5-panel compact layout)

In [ ]:
fig = plt.figure(figsize=(22, 5), facecolor=DARK_BG)
gs  = gridspec.GridSpec(1, 5, figure=fig, wspace=0.3,
                        left=0.04, right=0.98, top=0.88, bottom=0.15)

tns     = meta.get("f:xm_tns_fullname", "")
tns_str = f"  TNS: {tns}" if pd.notna(tns) and tns else ""
fig.suptitle(
    f"diaObjectId {OBJ_ID}{tns_str}  |  tag: {TAG_CONFIG}",
    fontsize=10, color=TEXT_COL, fontfamily="monospace", y=0.97
)

plot_lightcurve_flux(lc, ax=fig.add_subplot(gs[0, 0]))
plot_lightcurve_mag(lc,  ax=fig.add_subplot(gs[0, 1]))
plot_cutouts(cutouts_arr, band=meta.get("r:band", "?"),
             axes=[fig.add_subplot(gs[0, i+2]) for i in range(3)])

plt.show()

## 10 — Detailed 2×3 layout

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), facecolor=DARK_BG)
fig.patch.set_facecolor(DARK_BG)

zphot   = meta.get("f:xm_legacydr8_zphot", "—")
tns_str = f"  TNS: {tns}" if pd.notna(tns) and tns else ""
fig.suptitle(
    f"diaObjectId {OBJ_ID}{tns_str}  |  "
    f"RA={meta['r:ra']:.4f}°  Dec={meta['r:dec']:.4f}°  zphot={zphot}",
    fontsize=11, color=TEXT_COL, fontfamily="monospace", y=0.99
)

plot_lightcurve_flux(lc, ax=axes[0, 0])
plot_lightcurve_mag(lc,  ax=axes[0, 1])
plot_classifiers(meta,   ax=axes[0, 2])
plot_cutouts(cutouts_arr, band=meta.get("r:band", "?"),
             axes=[axes[1, 0], axes[1, 1], axes[1, 2]])

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 11 — Thumbnail grid for the whole tag

In [ ]:
N_COLS      = 6      # columns in the grid
MAX_OBJECTS = 36     # limit for display (None = all)

obj_ids_grid = ds.obj_ids[:MAX_OBJECTS] if MAX_OBJECTS else ds.obj_ids
n     = len(obj_ids_grid)
nrows = int(np.ceil(n / N_COLS))

fig, axes = plt.subplots(
    nrows, N_COLS,
    figsize=(N_COLS * 3, nrows * 3),
    facecolor=DARK_BG,
    squeeze=False,
)
fig.suptitle(
    f"{TAG_CONFIG}  —  {n} objects  (highlighted: {OBJ_ID})",
    fontsize=11, color=TEXT_COL, fontfamily="monospace", y=1.01
)

zscale = ZScaleInterval(contrast=0.25)

for i, oid in enumerate(obj_ids_grid):
    ax   = axes[i // N_COLS][i % N_COLS]
    row  = ds.get_meta(oid)
    arr  = ds.get_cutout_array(oid)
    ax.set_facecolor(PANEL_BG)

    if arr is not None:
        img = arr[0]   # Science stamp
        try:
            vmin, vmax = zscale.get_limits(img)
        except Exception:
            vmin, vmax = np.nanpercentile(img, [1, 99])
        ax.imshow(img, origin="lower", cmap="afmhot",
                  vmin=vmin, vmax=vmax, aspect="auto", interpolation="nearest")
        cy, cx = np.array(img.shape) / 2
        ax.plot(cx, cy, "+", color=HIGHLIGHT, ms=10, mew=1.5)
    else:
        ax.text(0.5, 0.5, "no\ncutout", ha="center", va="center",
                color=MUTED_COL, transform=ax.transAxes, fontsize=7)

    snn  = row.get("f:clf_snnSnVsOthers_score", float("nan"))
    snr  = row.get("r:snr", float("nan"))
    band = row.get("r:band", "?")
    tns_n = row.get("f:xm_tns_fullname", "")
    label_txt = f"#{i}  …{str(oid)[-6:]}\nSNN={snn:.2f}  SNR={snr:.0f}\n{band}"
    if pd.notna(tns_n) and tns_n:
        label_txt += f"\n{tns_n}"
    ax.text(0.02, 0.98, label_txt, transform=ax.transAxes, color=TEXT_COL,
            fontsize=5.5, va="top", fontfamily="monospace",
            bbox=dict(facecolor=DARK_BG, alpha=0.7, edgecolor="none", pad=1.5))

    border_col = HIGHLIGHT if oid == OBJ_ID else BORDER
    border_lw  = 2.5      if oid == OBJ_ID else 0.7
    for sp in ax.spines.values():
        sp.set_edgecolor(border_col)
        sp.set_linewidth(border_lw)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Hide unused cells
for j in range(n, nrows * N_COLS):
    axes[j // N_COLS][j % N_COLS].set_visible(False)

plt.tight_layout()
plt.show()

## 12 — Loop over all objects

Set `LOOP_PLOT_TYPE` to control what is displayed per object:  
`'overview'` · `'detail'` · `'lc_flux'` · `'lc_mag'` · `'cutouts'`

In [ ]:
LOOP_PLOT_TYPE = "overview"   # 'overview' | 'detail' | 'lc_flux' | 'lc_mag' | 'cutouts'
LOOP_MAX       = 5            # stop after N objects (None = all)
LOOP_SAVE      = False        # save each figure as PNG?

loop_ids = ds.obj_ids[:LOOP_MAX] if LOOP_MAX else ds.obj_ids

for i, oid in enumerate(loop_ids):
    row = ds.get_meta(oid)
    lc_i = ds.get_lightcurve(oid)
    arr_i = ds.get_cutout_array(oid)
    band_i = row.get("r:band", "?")
    tns_i  = row.get("f:xm_tns_fullname", "")
    tns_s  = f"  TNS: {tns_i}" if pd.notna(tns_i) and tns_i else ""

    print(f"[{i+1}/{len(loop_ids)}]  diaObjectId={oid}  SNN={row.get('f:clf_snnSnVsOthers_score', float('nan')):.3f}{tns_s}")

    if LOOP_PLOT_TYPE == "overview":
        fig = plt.figure(figsize=(22, 5), facecolor=DARK_BG)
        gs  = gridspec.GridSpec(1, 5, figure=fig, wspace=0.3,
                                left=0.04, right=0.98, top=0.88, bottom=0.15)
        fig.suptitle(f"diaObjectId {oid}{tns_s}  |  {TAG_CONFIG}",
                     fontsize=10, color=TEXT_COL, fontfamily="monospace", y=0.97)
        plot_lightcurve_flux(lc_i, ax=fig.add_subplot(gs[0, 0]))
        plot_lightcurve_mag(lc_i,  ax=fig.add_subplot(gs[0, 1]))
        plot_cutouts(arr_i, band=band_i,
                     axes=[fig.add_subplot(gs[0, k+2]) for k in range(3)])

    elif LOOP_PLOT_TYPE == "detail":
        fig, axes = plt.subplots(2, 3, figsize=(18, 9), facecolor=DARK_BG)
        fig.patch.set_facecolor(DARK_BG)
        fig.suptitle(f"diaObjectId {oid}{tns_s}  |  {TAG_CONFIG}",
                     fontsize=10, color=TEXT_COL, fontfamily="monospace", y=0.99)
        plot_lightcurve_flux(lc_i, ax=axes[0, 0])
        plot_lightcurve_mag(lc_i,  ax=axes[0, 1])
        plot_classifiers(row,       ax=axes[0, 2])
        plot_cutouts(arr_i, band=band_i, axes=[axes[1, 0], axes[1, 1], axes[1, 2]])
        plt.tight_layout(rect=[0, 0, 1, 0.97])

    elif LOOP_PLOT_TYPE == "lc_flux":
        fig, ax = plt.subplots(figsize=(9, 4), facecolor=DARK_BG)
        plot_lightcurve_flux(lc_i, ax=ax, title=f"Flux LC — {oid}")

    elif LOOP_PLOT_TYPE == "lc_mag":
        fig, ax = plt.subplots(figsize=(9, 4), facecolor=DARK_BG)
        plot_lightcurve_mag(lc_i, ax=ax, title=f"Mag LC — {oid}")

    elif LOOP_PLOT_TYPE == "cutouts":
        fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor=DARK_BG)
        plot_cutouts(arr_i, band=band_i, axes=list(axes))
        fig.suptitle(f"Cutouts — {oid}{tns_s}", color=TEXT_COL,
                     fontsize=10, fontfamily="monospace")

    if LOOP_SAVE:
        out = tag_dir / f"{oid}_{LOOP_PLOT_TYPE}.png"
        fig.savefig(out, dpi=120, bbox_inches="tight", facecolor=DARK_BG)

    plt.show()
    plt.close(fig)

## 13 — Light curve comparison across multiple objects

In [ ]:
# Pick the first N objects to compare
COMPARE_N = 8
compare_ids = ds.obj_ids[:COMPARE_N]

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=DARK_BG)
for ax in axes:
    ax.set_facecolor(PANEL_BG)

for oid in compare_ids:
    lc_c = ds.get_lightcurve(oid)
    if lc_c.empty:
        continue
    t0 = lc_c["r:midpointMjdTai"].min()
    for band, grp in lc_c.groupby("r:band"):
        col = BAND_COLORS.get(band, "gray")
        dt  = grp["r:midpointMjdTai"] - t0
        axes[0].errorbar(dt, grp["r:psfFlux"], yerr=grp["r:psfFluxErr"],
                         fmt="o-", ms=2, color=col, alpha=0.5, lw=0.8)
        mag, mag_err = flux_to_mag(grp["r:psfFlux"].values, grp["r:psfFluxErr"].values)
        valid = np.isfinite(mag)
        if valid.sum() > 0:
            axes[1].errorbar(dt.values[valid], mag[valid], yerr=mag_err[valid],
                             fmt="o-", ms=2, color=col, alpha=0.5, lw=0.8)

for ax, ylabel, title in zip(
    axes,
    ["psfFlux (nJy)", "mag AB"],
    ["Flux comparison", "Magnitude comparison"],
):
    ax.set_xlabel("Δ MJD TAI (days)", color=MUTED_COL)
    ax.set_ylabel(ylabel, color=MUTED_COL)
    ax.set_title(title, color=ACCENT)
    ax.tick_params(colors=MUTED_COL)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)

axes[1].invert_yaxis()
fig.suptitle(f"{TAG_CONFIG}  —  first {COMPARE_N} objects",
             fontsize=11, color=TEXT_COL, fontfamily="monospace", y=1.01)
plt.tight_layout()
plt.show()